# PySpark ETL


## Создаем Spark Session

In [1]:
import os
import warnings
from pathlib import Path
import pandas as pd

warnings.filterwarnings('ignore')

if 'JAVA_HOME' not in os.environ:
    candidate = '/usr/local/Cellar/openjdk@21/21.0.11/libexec/openjdk.jdk/Contents/Home'
    if Path(candidate).exists():
        os.environ['JAVA_HOME'] = candidate

DATA_DIR = Path.cwd() / 'output'
DATA_DIR.mkdir(exist_ok=True)

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('02-pyspark')
    .master('local[4]')
    .config('spark.sql.shuffle.partitions', 8)
    .config('spark.sql.repl.eagerEval.enabled', True)
    .config('spark.sql.execution.arrow.pyspark.enabled', True)
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.LocalFileSystem")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/26 21:24:22 WARN Utils: Your hostname, iMac-Ekaterina.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.86 instead (on interface en1)
26/04/26 21:24:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/26 21:24:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Cхема таблиц

In [3]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType, TimestampType

users_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("country", StringType(), True),
    StructField("signup_date", DateType(), True),
])

products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("user_id", DoubleType(), True),     # после будет IntegerType
    StructField("order_ts", StringType(), True), # после будет TimestampType
    StructField("status", StringType(), True),
])

order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("product_id", DoubleType(), True),    # после будет IntegerType
    StructField("quantity", IntegerType(), True),
])

### Чтение

In [4]:
users = spark.read.format("csv").option("header","true").schema(users_schema).load("./data/raw/users.csv")
products = spark.read.format("csv").option("header","true").schema(products_schema).load("./data/raw/products.csv")
orders = spark.read.format("csv").option("header","true").schema(orders_schema).load("./data/raw/orders.csv")
order_items = spark.read.format("csv").option("header","true").schema(order_items_schema).load("./data/raw/order_items.csv")

### Переобразование `product_id` и `user_id` в IntegerType, `order_ts` в TimestampType

In [5]:
from pyspark.sql.functions import col, to_timestamp

orders = orders.withColumn("user_id", col("user_id").cast(IntegerType()))
order_items = order_items.withColumn("product_id", col("product_id").cast(IntegerType()))

orders = orders.withColumn("order_ts", to_timestamp(col("order_ts"))) #преобразуем через функцию, чтобы даты в разных форматах считались как timestamp


### Посмотрим как выглядят таблицы

#### `users`

In [6]:
users.printSchema()
print("Число строк в таблице users:", users.count(), '\n')
users.show(10)


root
 |-- user_id: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



Число строк в таблице users: 5000 

+-------+-------+-----------+
|user_id|country|signup_date|
+-------+-------+-----------+
|      1|     RU| 2024-05-24|
|      2|     UZ| 2024-01-25|
|      3|     KZ| 2025-04-03|
|      4|     RS| 2025-05-14|
|      5|     RS| 2025-07-18|
|      6|     UZ| 2025-11-25|
|      7|     RU| 2024-12-29|
|      8|     KZ| 2025-08-29|
|      9|     DE| 2025-06-07|
|     10|     RU| 2024-07-21|
+-------+-------+-----------+
only showing top 10 rows


#### `products`

In [7]:
products.printSchema()
print("Число строк в таблице orders:", products.count(), '\n')
products.show(10)

root
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)

Число строк в таблице orders: 1500 

+----------+-----------+------+
|product_id|   category| price|
+----------+-----------+------+
|         1|       home| 30.58|
|         2|      books| 270.8|
|         3|    fashion|532.97|
|         4|     sports|528.26|
|         5|       home| 64.71|
|         6|       toys|428.59|
|         7|     sports|104.29|
|         8|      books|347.78|
|         9|    grocery|171.58|
|        10|electronics| 41.95|
+----------+-----------+------+
only showing top 10 rows


#### `orders`

In [8]:
orders.printSchema()
print("Число строк в таблице orders:", orders.count(), '\n')
orders.show(10)


root
 |-- order_id: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)

Число строк в таблице orders: 25500 

+-----------+-------+-------------------+---------+
|   order_id|user_id|           order_ts|   status|
+-----------+-------+-------------------+---------+
|ORD-0006914|   1987|2026-01-23 10:12:25|  shipped|
|ORD-0011125|   2058|2026-01-25 07:17:11|completed|
|ORD-0007229|   2320|2026-01-29 09:00:31|completed|
|ORD-0002765|   1394|2026-01-23 02:54:57|completed|
|ORD-0023217|   NULL|2026-01-02 00:06:37|  shipped|
|ORD-0017270|   1428|2026-01-19 21:51:59|cancelled|
|ORD-0003074|   2613|2026-01-11 08:43:04|  shipped|
|ORD-0008185|   2171|2026-01-24 14:51:30|completed|
|ORD-0002596|   3175|2026-01-04 13:18:50|  pending|
|ORD-0005484|   4691|2026-01-31 14:21:41|completed|
+-----------+-------+-------------------+---------+
only showing top 10 rows


#### `order_items`

In [9]:

order_items.printSchema()
print("Число строк в таблице orders:", order_items.count(), '\n')
order_items.show(10)


root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)

Число строк в таблице orders: 60864 

+-----------+----------+--------+
|   order_id|product_id|quantity|
+-----------+----------+--------+
|ORD-0020674|       327|       1|
|ORD-0003276|      1446|       3|
|ORD-0015563|      1329|       1|
|ORD-0018816|       657|       3|
|ORD-0010236|       284|       2|
|ORD-0000617|        30|       4|
|ORD-0012522|      1356|       3|
|ORD-0009095|      1301|       1|
|ORD-0001203|      1151|       4|
|ORD-0018515|      1152|       1|
+-----------+----------+--------+
only showing top 10 rows


## Очистка данных

1. удалить дубликаты заказов по `order_id`;
2. удалить или отфильтровать строки с `null` в ключевых полях;
3. удалить строки с `quantity <= 0`;
4. исключить отменённые заказы, где `status = 'cancelled'`;
5. выделить поле с датой заказа;
6. исключить строки `order_items`, для которых нет товара в `products`;
7. исключить строки `orders`, для которых нет пользователя в `users`.

In [ ]:
from pyspark.sql.functions import to_date, broadcast

orders_cleaned = (
    orders
    .dropDuplicates(["order_id"]) 
    .filter(
        col("order_id").isNotNull() &
        col("user_id").isNotNull()
    ) 
    .filter(col("status") != "cancelled") 
    .withColumn("date", to_date("order_ts")) 
)

order_items_cleaned = (
    order_items
    .filter(
        col("product_id").isNotNull() &
        col("order_id").isNotNull()
    ) 
    .filter(col("quantity") > 0) 
)

users_cleaned = users.filter(col("user_id").isNotNull())

products_cleaned = products.filter(
    col("product_id").isNotNull() &
    col("price").isNotNull()
) 

products_cleaned = products_cleaned.fillna({"category": "-"})

orders_cleaned = (orders_cleaned
    .join(broadcast(users_cleaned), "user_id", "inner")
    .select("order_id", "user_id", "order_ts", "status", "date")
)

order_items_cleaned = (order_items_cleaned
                       .join(broadcast(products_cleaned), "product_id", "inner")
                       .select("order_id", "product_id", "quantity")
)

order_items_cleaned = (order_items_cleaned
                       .join(broadcast(orders_cleaned), "order_id", "inner")
                       .select("order_id", "product_id", "quantity")
)


In [88]:
print("ДАННЫЕ ДО ОТЧИСТКИ")
print(f"users: {users.count()}")
print(f"products: {products.count()}")
print(f"orders: {orders.count()}")
print(f"order_items: {order_items.count()}")

print("\nДАННЫЕ ПОСЛЕ ОТЧИСТКИ")
print(f"users_cleaned: {users_cleaned.count()}")
print(f"products_cleaned: {products_cleaned.count()}")
print(f"orders_cleaned: {orders_cleaned.count()}")
print(f"order_items_cleaned: {order_items_cleaned.count()}")


ДАННЫЕ ДО ОТЧИСТКИ
users: 5000
products: 1500
orders: 25500
order_items: 60864

ДАННЫЕ ПОСЛЕ ОТЧИСТКИ
users_cleaned: 5000
products_cleaned: 1500
orders_cleaned: 20638
order_items_cleaned: 48746


In [89]:
print("\nтаблица users")
users_cleaned.show(5)
print("\nтаблица products")
products_cleaned.show(5)
print("\nтаблица orders")
orders_cleaned.show(5)
print("\nтаблица order_items")
order_items_cleaned.show(5)


таблица users
+-------+-------+-----------+
|user_id|country|signup_date|
+-------+-------+-----------+
|      1|     RU| 2024-05-24|
|      2|     UZ| 2024-01-25|
|      3|     KZ| 2025-04-03|
|      4|     RS| 2025-05-14|
|      5|     RS| 2025-07-18|
+-------+-------+-----------+
only showing top 5 rows

таблица products
+----------+--------+------+
|product_id|category| price|
+----------+--------+------+
|         1|    home| 30.58|
|         2|   books| 270.8|
|         3| fashion|532.97|
|         4|  sports|528.26|
|         5|    home| 64.71|
+----------+--------+------+
only showing top 5 rows

таблица orders
+-----------+-------+-------------------+---------+----------+
|   order_id|user_id|           order_ts|   status|      date|
+-----------+-------+-------------------+---------+----------+
|ORD-0000001|   3699|2026-01-06 08:35:54|completed|2026-01-06|
|ORD-0000002|    801|2026-01-13 06:17:03|completed|2026-01-13|
|ORD-0000003|   3594|2026-01-27 06:12:22|completed|2026-0

## Проверки качества данных

- после очистки `order_id` уникален;
- в итоговых витринах нет `null` в ключевых полях;
- `revenue >= 0`;
- итоговые витрины не пустые;
- строки `order_items` без товара не попадают в итоговые расчёты;
- строки `orders` без пользователя не попадают в итоговые расчёты.

In [90]:
# после очистки `order_id` уникален
is_distinct_order_id = orders_cleaned.groupBy("order_id").count().filter(col("count") > 1).count()
assert is_distinct_order_id == 0, f"Ошибка: Найдено {is_distinct_order_id} дубликатов order_id "


# строки order_items без товара не попадают в итоговые расчёты
order_items_product_id = order_items_cleaned.select("product_id").distinct()
products_product_id = products_cleaned.select("product_id").distinct()
missing_products = order_items_product_id.join(products_product_id, "product_id", "left_anti")
assert missing_products.count() == 0, f"Ошибка: Найдено {missing_products.count()} null значений product_id в таблице order_items"


# строки orders без пользователя не попадают в итоговые расчёты
orders_user_id = orders_cleaned.select("user_id").distinct()
users_user_id = users_cleaned.select("user_id").distinct()
missing_users = orders_user_id.join(users_user_id, "user_id", "left_anti")
assert missing_users.count() == 0, f"Ошибка: Найдено {missing_users.count()} null значений user_id в таблице orders"

## Витрины данных

In [91]:
sales_df = (
    orders_cleaned
    .join(broadcast(users_cleaned).alias("u"), "user_id", "inner")
    .join(order_items_cleaned.alias("oi"), "order_id", "inner")
    .join(broadcast(products_cleaned).alias("p"), "product_id", "inner")
    .withColumn("revenue", col("quantity") * col("price"))
)

print("Отчищенный датасет sales")
print("\nЧисло строк в датасете:", sales_df.count())
sales_df

Отчищенный датасет sales

Число строк в датасете: 48746


product_id,order_id,user_id,order_ts,status,date,country,signup_date,quantity,category,price,revenue
606,ORD-0000001,3699,2026-01-06 08:35:54,completed,2026-01-06,GE,2024-11-20,4,sports,352.42,1409.68
679,ORD-0000002,801,2026-01-13 06:17:03,completed,2026-01-13,RU,2024-03-31,2,books,329.36,658.72
557,ORD-0000002,801,2026-01-13 06:17:03,completed,2026-01-13,RU,2024-03-31,4,sports,486.7,1946.8
284,ORD-0000002,801,2026-01-13 06:17:03,completed,2026-01-13,RU,2024-03-31,2,fashion,19.34,38.68
1365,ORD-0000002,801,2026-01-13 06:17:03,completed,2026-01-13,RU,2024-03-31,1,toys,565.54,565.54
619,ORD-0000003,3594,2026-01-27 06:12:22,completed,2026-01-27,AM,2024-11-21,3,grocery,389.61,1168.83
567,ORD-0000004,251,2026-01-14 05:18:34,completed,2026-01-14,DE,2024-04-19,1,books,374.39,374.39
215,ORD-0000004,251,2026-01-14 05:18:34,completed,2026-01-14,DE,2024-04-19,3,toys,487.6,1462.8000000000002
178,ORD-0000004,251,2026-01-14 05:18:34,completed,2026-01-14,DE,2024-04-19,1,grocery,573.0,573.0
1398,ORD-0000004,251,2026-01-14 05:18:34,completed,2026-01-14,DE,2024-04-19,2,toys,535.84,1071.68


### Витрина 1: daily_sales_mart

Проверки: 
- в итоговых витринах нет `null` в ключевых полях;
- `revenue >= 0`;
- итоговые витрины не пустые;

In [98]:
from pyspark.sql.functions import count_distinct, sum


daily_sales_mart = (sales_df
                    .groupBy("date", "country")
                    .agg(
                        count_distinct("order_id").alias("orders_cnt"),
                        count_distinct("user_id").alias("buyers_cnt"),
                        sum("quantity").alias("items_cnt"),
                        sum("revenue").alias("revenue")
                    )
).orderBy("date", "country")

print("Витрина daily_sales_mart")
daily_sales_mart.show(10)

# проверки качества данных

null_count_date = daily_sales_mart.filter(col("date").isNull()).count()
assert null_count_date == 0, f"Ошибка: В {daily_sales_mart} найдено {null_count_date} null-значений в колонке date"

null_count_country = daily_sales_mart.filter(col("country").isNull()).count()
assert null_count_country == 0, f"Ошибка: В {daily_sales_mart} найдено {null_count_country} null-значений в колонке country"

invailid_revenue = daily_sales_mart.filter(col("revenue") < 0).count()
assert invailid_revenue == 0, f"Ошибка: revenue < 0"

assert daily_sales_mart.count() > 0, f"Ошибка: итоговая витрина daily_sales_mart пустая"

# запись
output_daily_sales_mart = DATA_DIR / 'daily_sales_mart'
(
    daily_sales_mart
    .repartition('date')
    .write
    .mode('overwrite')
    .partitionBy('date')
    .parquet(str(output_daily_sales_mart))
)

# чтение
print("\nВитрина daily_sales_mart из parquet")
daily_sales_mart_parquet_df = spark.read.parquet(str(output_daily_sales_mart))
daily_sales_mart_parquet_df.printSchema()
daily_sales_mart_parquet_df.show(10)

Витрина daily_sales_mart
+----------+-------+----------+----------+---------+------------------+
|      date|country|orders_cnt|buyers_cnt|items_cnt|           revenue|
+----------+-------+----------+----------+---------+------------------+
|2026-01-01|     AM|        82|        77|      476|174785.72000000006|
|2026-01-01|     DE|        81|        73|      434|153209.10000000003|
|2026-01-01|     GE|        82|        79|      513|174861.50999999998|
|2026-01-01|     KZ|        88|        77|      539|196308.49000000008|
|2026-01-01|     PL|        68|        64|      467|         170140.18|
|2026-01-01|     RS|        76|        68|      468| 172023.7199999999|
|2026-01-01|     RU|        76|        71|      502|         190157.37|
|2026-01-01|     UZ|        81|        75|      529|179385.71000000005|
|2026-01-02|     AM|        80|        78|      522|182476.04000000007|
|2026-01-02|     DE|        75|        71|      479|165832.62999999998|
+----------+-------+----------+--------


Витрина daily_sales_mart из parquet
root
 |-- country: string (nullable = true)
 |-- orders_cnt: long (nullable = true)
 |-- buyers_cnt: long (nullable = true)
 |-- items_cnt: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- date: date (nullable = true)

+-------+----------+----------+---------+------------------+----------+
|country|orders_cnt|buyers_cnt|items_cnt|           revenue|      date|
+-------+----------+----------+---------+------------------+----------+
|     KZ|        86|        81|      529|166931.43000000002|2026-01-26|
|     RU|        85|        78|      539|          194790.7|2026-01-26|
|     RS|        81|        77|      552|189520.49000000002|2026-01-26|
|     AM|        84|        79|      580|202834.13999999987|2026-01-26|
|     DE|        76|        73|      474|172780.33000000007|2026-01-26|
|     GE|        78|        72|      517|         184419.39|2026-01-26|
|     UZ|        81|        76|      520|173066.32999999996|2026-01-26|
|     

### Витрина 2: category_sales_mart

In [99]:
category_sales_mart = (sales_df
                    .groupBy("date", "category")
                    .agg(
                        count_distinct("order_id").alias("orders_cnt"),
                        sum("quantity").alias("items_cnt"),
                        sum("revenue").alias("revenue")
                    )
).orderBy("date", "category")

print("Витрина category_sales_mart")
category_sales_mart.show(10)

# проверки качества данных
null_count_date = category_sales_mart.filter(col("date").isNull()).count()
assert null_count_date == 0, f"Ошибка: В {category_sales_mart} найдено {null_count_date} null-значений в колонке date"

null_count_category = category_sales_mart.filter(col("category").isNull()).count()
assert null_count_category == 0, f"Ошибка: В {category_sales_mart} найдено {null_count_category} null-значений в колонке category"

invailid_revenue = category_sales_mart.filter(col("revenue") < 0).count()
assert invailid_revenue == 0, f"Ошибка: revenue < 0"

assert category_sales_mart.count() > 0, f"Ошибка: итоговая витрина category_sales_mart пустая"

# запись
output_category_sales_mart = DATA_DIR / 'category_sales_mart'
(
    category_sales_mart
    .repartition('date')
    .write
    .mode('overwrite')
    .partitionBy('date')
    .parquet(str(output_category_sales_mart))
)

# чтение
print("\nВитрина category_sales_mart из parquet")
category_sales_mart_parquet_df = spark.read.parquet(str(output_category_sales_mart))
category_sales_mart_parquet_df.printSchema()
category_sales_mart_parquet_df.show(10)

Витрина category_sales_mart
+----------+-----------+----------+---------+------------------+
|      date|   category|orders_cnt|items_cnt|           revenue|
+----------+-----------+----------+---------+------------------+
|2026-01-01|          -|        25|       66|24593.269999999997|
|2026-01-01|     beauty|       155|      435|162559.56999999992|
|2026-01-01|      books|       159|      446|156558.96999999994|
|2026-01-01|electronics|       168|      472| 160259.3200000001|
|2026-01-01|    fashion|       186|      528| 195540.0400000001|
|2026-01-01|    grocery|       169|      465|161902.46999999994|
|2026-01-01|       home|       183|      515|189488.27000000002|
|2026-01-01|     sports|       149|      436| 152272.9899999999|
|2026-01-01|       toys|       200|      565|207696.90000000002|
|2026-01-02|          -|        31|       78|           31186.7|
+----------+-----------+----------+---------+------------------+
only showing top 10 rows



Витрина category_sales_mart из parquet
root
 |-- category: string (nullable = true)
 |-- orders_cnt: long (nullable = true)
 |-- items_cnt: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- date: date (nullable = true)

+-----------+----------+---------+------------------+----------+
|   category|orders_cnt|items_cnt|           revenue|      date|
+-----------+----------+---------+------------------+----------+
|    grocery|       184|      489|172438.99000000005|2026-01-29|
|       home|       168|      455|153613.87999999998|2026-01-29|
|      books|       171|      500|180353.57000000004|2026-01-29|
|       toys|       168|      487|178250.58999999985|2026-01-29|
|    fashion|       168|      438|         163989.88|2026-01-29|
|     sports|       176|      551|         174780.71|2026-01-29|
|          -|        29|       75|28218.170000000002|2026-01-29|
|     beauty|       146|      416|149264.22000000003|2026-01-29|
|electronics|       174|      537|184122.600000

### Витрина 3: user_sales_mart

In [100]:
sales_df.createOrReplaceTempView('sales')

user_sales_mart = spark.sql("""
    SELECT 
        user_id,
        country,
        COUNT(DISTINCT order_id) AS orders_cnt,
        ROUND(SUM(revenue), 2) AS revenue,
        MIN(order_ts) AS first_order_ts,
        MAX(order_ts) AS last_order_ts
    FROM sales
    GROUP BY user_id, country
    ORDER BY user_id
""")

print("Витрина user_sales_mart")
user_sales_mart.show(10)


# проверки качества данных
null_count_user_id = user_sales_mart.filter(col("user_id").isNull()).count()
assert null_count_user_id == 0, f"Ошибка: В {user_sales_mart} найдено {null_count_user_id} null-значений в колонке user_id"

null_count_country = user_sales_mart.filter(col("country").isNull()).count()
assert null_count_country == 0, f"Ошибка: В {user_sales_mart} найдено {null_count_country} null-значений в колонке country"

invailid_revenue = user_sales_mart.filter(col("revenue") < 0).count()
assert invailid_revenue == 0, f"Ошибка: revenue < 0"

assert user_sales_mart.count() > 0, f"Ошибка: итоговая витрина user_sales_mart пустая"

# запись
output_user_sales_mart = DATA_DIR / 'user_sales_mart'
(
    user_sales_mart
    .write
    .mode('overwrite')
    .parquet(str(output_user_sales_mart))
)

# чтение
print("\nВитрина user_sales_mart из parquet")
user_sales_mart_parquet_df = spark.read.parquet(str(output_user_sales_mart))
user_sales_mart_parquet_df.printSchema()
user_sales_mart_parquet_df.show(10)

Витрина user_sales_mart
+-------+-------+----------+--------+-------------------+-------------------+
|user_id|country|orders_cnt| revenue|     first_order_ts|      last_order_ts|
+-------+-------+----------+--------+-------------------+-------------------+
|      1|     RU|         5| 7561.61|2026-01-05 02:30:34|2026-01-22 07:40:02|
|      2|     UZ|         3| 5357.15|2026-01-01 18:10:32|2026-01-20 17:50:45|
|      3|     KZ|         4| 10419.6|2026-01-18 09:22:09|2026-01-22 19:21:31|
|      4|     RS|         4| 5862.27|2026-01-18 10:17:52|2026-01-31 09:43:49|
|      5|     RS|         2| 2502.04|2026-01-02 17:24:38|2026-01-23 10:24:07|
|      6|     UZ|         4| 5524.02|2026-01-14 04:39:33|2026-01-29 17:34:45|
|      7|     RU|        11|23379.44|2026-01-01 22:57:49|2026-01-30 13:54:46|
|      8|     KZ|         3| 9787.54|2026-01-07 23:37:29|2026-01-25 09:41:12|
|      9|     DE|         1|  682.37|2026-01-06 09:45:29|2026-01-06 09:45:29|
|     10|     RU|         3| 8555.82|202

## performance-разбор

In [101]:
print("Разбор производительности для daily_sales_mart:")
daily_sales_mart.explain("formatted")

Разбор производительности для daily_sales_mart:
== Physical Plan ==
AdaptiveSparkPlan (66)
+- Sort (65)
   +- Exchange (64)
      +- HashAggregate (63)
         +- Exchange (62)
            +- HashAggregate (61)
               +- HashAggregate (60)
                  +- Exchange (59)
                     +- HashAggregate (58)
                        +- Expand (57)
                           +- Project (56)
                              +- BroadcastHashJoin Inner BuildRight (55)
                                 :- Project (51)
                                 :  +- SortMergeJoin Inner (50)
                                 :     :- Project (20)
                                 :     :  +- BroadcastHashJoin Inner BuildRight (19)
                                 :     :     :- Project (15)
                                 :     :     :  +- BroadcastHashJoin Inner BuildRight (14)
                                 :     :     :     :- Project (10)
                                 :     :     :

### Где в пайплайне возникает shuffle?
Shuffle (Exchange) происходит несколько раз.  
Например, в строчке 
```
(6) Exchange
Input [7]: [order_id#6, first#25851, valueSet#25852, first#25853, valueSet#25854, first#25855, valueSet#25856]
Arguments: hashpartitioning(order_id#6, 8), ENSURE_REQUIREMENTS, [plan_id=226970]
```
Shuffle возникает при агрегации по order_id. 

Вообще Shuffle возникает при операциях groupBy и join, так как Spark должен распределить данные между экзекьюторами по ключам.

### Почему в одном из join используется broadcast?
```
(13) BroadcastExchange
Input [1]: [user_id#0]
Arguments: HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=226976]

(14) BroadcastHashJoin
Left keys [1]: [user_id#25809]
Right keys [1]: [user_id#0]
Join type: Inner
Join condition: None
```
Broadcast join используется для маленьких таблиц (в данном случае users), что позволяет избежать shuffle.

```
(49) Sort
Input [3]: [order_id#10, product_id#16, quantity#12]
Arguments: [order_id#10 ASC NULLS FIRST], false, 0

(50) SortMergeJoin
Left keys [1]: [order_id#6]
Right keys [1]: [order_id#10]
Join type: Inner
Join condition: None
```

Для больших таблиц (в данном случае orders) используется SortMergeJoin, требующий shuffle и сортировки данных.

### Зачем используется partitionBy("date")?

Партиционирование по дате нужно, чтобы при чтении Spark только мог загрузить нужные партиции, что значительно ускоряет запросы.
Плюс можно настроить пайплайн, чтобы после определенной даты удалять быстро данные. 
 

In [102]:
spark.stop()